# Interactive Shape Blend Splines Demo

This notebook demonstrates the **non-rational** Shape Blend Spline (SBS) formulation:

$$\mathbf{C}(t)=\sum_{j=0}^{k-1} W_j(t)\,\mathbf{S}_j(t).$$

- $\mathbf{S}_j(t)$ are whole parametric shapes evaluated at the same global parameter $t$
- $W_j(t)$ are partition-of-unity weights built directly from smooth **polynomial** step-function differences
- there is **no rational denominator**
- `alpha` (α) controls the transition from square-like local edge behaviour to ellipse-like global smoothing

The four sections below walk through the SBS story: corrected closed curves from edge lines, open locality-aware design, non-rational weight plots, and a control-point workflow.

In [ ]:
# Install dependencies on Google Colab only
import subprocess
import sys

def colab_install():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "numpy", "matplotlib", "ipywidgets", "--quiet"])
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "git+https://github.com/QL-UoHull/Shape-Blend-Splines.git",
                           "--quiet"])

colab_install()

In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import numpy as np

from shape_blend_splines import (
    ControlPointSpline,
    PeriodicShapeBlendSpline,
    ShapeBlendSpline,
    ShapeBlender,
)
from shape_blend_splines.basis import blend_weights
from shape_blend_splines.shapes import (
    circle_arc,
    ellipse_arc,
    from_control_points,
    line_segment,
    rectangle_arc,
    star_arc,
    superellipse_arc,
)

## 1) Closed SBS curve — 4 control points / 4 edge lines

**The idea in one sentence:** supply 4 corners of a square, form the 4 straight edges as the constituent shapes, phase-align those edges with the periodic parameter, and let a *closed periodic* SBS blend them with the non-rational locality parameter \(\alpha\) — giving the intended square-like \(\rightarrow\) transitional \(\rightarrow\) ellipse-like family from the same input geometry.

### Setup

```
corner 3 (-1, 1) ─── edge 2 ─── corner 2 (1, 1)
     │                                │
  edge 3                           edge 1
     │                                │
corner 0 (-1,-1) ─── edge 0 ─── corner 1 (1,-1)
```

Each edge is still a `line_segment`, but it is re-parameterised so that the **midpoint of the edge** is sampled when the corresponding periodic SBS basis reaches its centre. The basis itself is built directly from telescoping smooth polynomial steps, so the closed curve remains non-rational throughout.

In [ ]:
# ── Square corners and phase-aligned edges ────────────────
corners = [(-1.0, -1.0), (1.0, -1.0), (1.0, 1.0), (-1.0, 1.0)]  # CCW
edge_labels = ["Edge 0 (bottom)", "Edge 1 (right)", "Edge 2 (top)", "Edge 3 (left)"]
corner_labels = ["SW", "SE", "NE", "NW"]
centers = np.linspace(0.0, 1.0, len(corners), endpoint=False)

def closed_edge_cycle_shapes(corners):
    shapes = []
    for j, center in enumerate(centers):
        p0 = corners[j]
        p1 = corners[(j + 1) % len(corners)]

        def edge_shape(t, p0=p0, p1=p1, center=center):
            local_t = np.mod(np.asarray(t) - center + 0.5, 1.0)
            return line_segment(local_t, p0=p0, p1=p1)

        shapes.append(edge_shape)
    return shapes

edges = closed_edge_cycle_shapes(corners)
t_sq = np.linspace(0.0, 1.0, 600, endpoint=False)  # endpoint=False for closed curve

def square_sbs_curve(alpha=2.0):
    """Evaluate the 4-edge closed SBS curve for a given locality alpha."""
    return PeriodicShapeBlendSpline(edges, t_centers=centers, locality=alpha).evaluate(t_sq)

def square_sbs_weights(alpha=2.0):
    """Return the (4, m) weight matrix for the 4-edge SBS."""
    return PeriodicShapeBlendSpline(edges, t_centers=centers, locality=alpha).weights_at(t_sq)

# Reference square outline for plotting
sq_corners = np.array(corners + [corners[0]])  # closed polygon

print("4 square corners defined.")
print("Phase-aligned edge line segments created.")
print("Ready to evaluate corrected closed periodic SBS.")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
colors = plt.cm.tab10.colors

# Light reference edges
for j, (edge_fn, label) in enumerate(zip(edges, edge_labels)):
    sp = edge_fn(t_sq)
    ax.plot(sp[:, 0], sp[:, 1], "--", lw=1.0, alpha=0.3, color=colors[j])

# Square outline
ax.plot(sq_corners[:, 0], sq_corners[:, 1], ":", lw=1.0, color="gray", alpha=0.4)

# Corner markers
cx = [c[0] for c in corners]
cy = [c[1] for c in corners]
ax.scatter(cx, cy, s=80, zorder=5, color="dimgray")
for (x, y), lbl in zip(corners, corner_labels):
    ax.annotate(lbl, (x, y), textcoords="offset points", xytext=(6, 4), fontsize=9, color="dimgray")

# Closed SBS curve (square-like rounded case)
pts = square_sbs_curve(alpha=2.0)
ax.plot(pts[:, 0], pts[:, 1], color="steelblue", lw=2.8, label="Closed SBS (α=2.0)")

ax.set_aspect("equal")
ax.set_title("4 corners → 4 edge lines → Closed SBS curve")
ax.legend(fontsize=9)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
configs = [
    (2.0, "Square-like rounded shape"),
    (1.2, "Rounded-square transition"),
    (0.8, "Intermediate rounded shape"),
    (0.6, "Soft transitional blend"),
    (0.4, "Ellipse-like smooth shape"),
    (0.3, "Near-ellipse limit"),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))

for ax, (alpha, label) in zip(axes.flat, configs):
    pts = square_sbs_curve(alpha=alpha)
    ax.plot(pts[:, 0], pts[:, 1], color="steelblue", lw=2.5)
    ax.plot(sq_corners[:, 0], sq_corners[:, 1], ":", lw=0.8, color="gray", alpha=0.5)
    ax.scatter([c[0] for c in corners], [c[1] for c in corners],
               s=40, zorder=5, color="dimgray")
    ax.set_aspect("equal")
    ax.set_title(f"{label}\n\u03b1={alpha}", fontsize=9)
    ax.grid(alpha=0.2)

fig.suptitle("Closed SBS from 4 edge lines: non-rational locality sweep", y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

## 2) Open SBS sequence — control-point polygon with locality

An open (non-closed) SBS blends an ordered sequence of edges along a control-point polygon.  Varying the locality parameter α shows the transition between:

- **small α → global, smooth curve** that averages all edges simultaneously, and
- **large α → near-piecewise-linear** curve that hugs each individual edge closely.

The SBS result passes smoothly through the control-point region without oscillation, unlike polynomial interpolation.

In [ ]:
# Open control-point polygon (5 corners, 4 edges)
open_corners = np.array([
    [-2.0,  0.0],
    [-1.0,  1.5],
    [ 0.5,  0.3],
    [ 1.5,  1.5],
    [ 2.5,  0.0],
], dtype=float)

open_edges = [
    partial(line_segment, p0=tuple(open_corners[i]), p1=tuple(open_corners[i+1]))
    for i in range(len(open_corners) - 1)
]
open_labels_e = [f"Edge {i}" for i in range(len(open_edges))]

t_open = np.linspace(0.0, 1.0, 600)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
alphas = [0.5, 2.0, 8.0]
ec = plt.cm.Set2.colors

for ax, alpha in zip(axes, alphas):
    sbs = ShapeBlendSpline(open_edges, locality=alpha)
    pts = sbs.evaluate(t_open)

    # Light reference edges
    for j, edge_fn in enumerate(open_edges):
        sp = edge_fn(t_open)
        ax.plot(sp[:, 0], sp[:, 1], "--", lw=1.0, alpha=0.3, color=ec[j % len(ec)])

    # Control polygon
    ax.plot(open_corners[:, 0], open_corners[:, 1], "o--",
            color="tomato", lw=1.2, ms=5, alpha=0.7, label="Control polygon")

    # SBS curve
    ax.plot(pts[:, 0], pts[:, 1], color="steelblue", lw=2.5, label=f"SBS α={alpha}")
    ax.set_aspect("equal")
    ax.set_title(f"Open SBS, α={alpha}", fontsize=10)
    ax.grid(alpha=0.2)

axes[0].set_ylabel("y")
for ax in axes:
    ax.set_xlabel("x")
    ax.legend(fontsize=8)

fig.suptitle("Open SBS: same control polygon, varying locality α", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## 3) Interactive: locality sweep for the corrected 4-point closed family

The cell below is **interactive** when run in Colab or Jupyter (requires `ipywidgets`). It shows both the direct **SBS basis curves** $W_j(t)$ and the resulting **closed SBS curve** updating in real time as you move the locality slider.

*Static fallback:* the grid immediately below is pre-rendered for viewing on GitHub or any non-widget renderer.

In [ ]:
# ── Static fallback grid ────────────────────────────
fallback_alphas = [2.0, 1.2, 0.8, 0.6, 0.4, 0.3]
colors4 = ["#e15759", "#4e79a7", "#76b7b2", "#f28e2b"]

fig, axes = plt.subplots(2, 6, figsize=(18, 6.5))
for col, alpha in enumerate(fallback_alphas):
    ax_w = axes[0, col]
    ax_c = axes[1, col]

    sbs = PeriodicShapeBlendSpline(edges, t_centers=centers, locality=alpha)
    W = sbs.weights_at(t_sq)
    pts = sbs.evaluate(t_sq)

    for j in range(4):
        ax_w.plot(t_sq, W[j], color=colors4[j], lw=1.5, label=edge_labels[j])
    ax_w.set_ylim(-0.05, 1.05)
    ax_w.set_title(f"\u03b1={alpha}", fontsize=7)
    ax_w.set_xlabel("t", fontsize=7)
    if col == 0:
        ax_w.set_ylabel("$W_j(t)$", fontsize=8)
        ax_w.legend(fontsize=6, loc="upper right")

    ax_c.plot(pts[:, 0], pts[:, 1], color="steelblue", lw=2.2)
    ax_c.plot(sq_corners[:, 0], sq_corners[:, 1], ":", lw=0.8, color="gray", alpha=0.5)
    ax_c.scatter([c[0] for c in corners], [c[1] for c in corners],
                 s=20, zorder=5, color="dimgray")
    ax_c.set_aspect("equal")
    if col == 0:
        ax_c.set_ylabel("y", fontsize=8)
    ax_c.set_xlabel("x", fontsize=8)

fig.suptitle("Static fallback: direct SBS basis functions (top) and closed curve (bottom)",
             y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Interactive widget (requires ipywidgets) ────────────
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    _alpha_slider = widgets.FloatSlider(value=0.8, min=0.3, max=3.0, step=0.1,
                                        description="α (locality):",
                                        style={"description_width": "110px"},
                                        layout=widgets.Layout(width="380px"))

    _out = widgets.Output()

    def _update(_change=None):
        alpha = _alpha_slider.value
        sbs = PeriodicShapeBlendSpline(edges, t_centers=centers, locality=alpha)
        W = sbs.weights_at(t_sq)
        pts = sbs.evaluate(t_sq)

        with _out:
            clear_output(wait=True)
            fig, (ax_w, ax_c) = plt.subplots(1, 2, figsize=(12, 4.5))
            colors4 = ["#e15759", "#4e79a7", "#76b7b2", "#f28e2b"]
            for j in range(4):
                ax_w.plot(t_sq, W[j], color=colors4[j], lw=2.0,
                          label=f"$W_{j}$")
            ax_w.set_ylim(-0.05, 1.05)
            ax_w.set_xlabel("t"); ax_w.set_ylabel("$W_j(t)$")
            ax_w.set_title(f"Direct SBS basis weights  (α={alpha:.1f})")
            ax_w.legend(fontsize=9, loc="upper right")
            ax_w.grid(alpha=0.2)

            ax_c.plot(pts[:, 0], pts[:, 1], color="steelblue", lw=2.8,
                      label="Closed SBS")
            ax_c.plot(sq_corners[:, 0], sq_corners[:, 1],
                      ":", lw=0.8, color="gray", alpha=0.5)
            ax_c.scatter([c[0] for c in corners], [c[1] for c in corners],
                         s=60, zorder=5, color="dimgray")
            ax_c.set_aspect("equal")
            ax_c.set_title("Resulting closed SBS curve")
            ax_c.set_xlabel("x"); ax_c.set_ylabel("y")
            ax_c.grid(alpha=0.2)
            ax_c.legend(fontsize=9)
            plt.tight_layout()
            plt.show()

    _alpha_slider.observe(_update, names="value")

    _controls = widgets.VBox([_alpha_slider])
    display(widgets.HBox([_controls, _out]))
    _update()  # initial render

except ImportError:
    print("ipywidgets not available — see the static grid above for the visualisation.")
    print("Install with:  pip install ipywidgets")

## 4) Numerical verification: B-spline as a step-function difference

The theory document `docs/theory.md` establishes that each degree-$p$ B-spline basis function can be written as a **difference of two smooth step functions** (in the cumulative/divided-difference sense).  Concretely, for the SBS step-function family used in this package:

$$B_{a,b}(t) = S_b(t) - S_a(t),$$

where $S_c(t) = 1 - T_n\!\left(\tfrac{t-c+\sigma}{2\sigma}\right)$ is the falling smooth step at $c$.

The cell below verifies this **numerically** in two ways:

1. **Exact identity for `sbs_basis`:** confirms the implementation `sbs_basis(t,a,b)` matches the manual step-difference calculation to machine precision.
2. **Structural parallel with cubic B-splines:** shows that for a cubic ($p=3$) uniform clamped B-spline, the Cox–de Boor values and the `sbs_basis` approximation at the same support endpoints share the same bell-shaped, non-negative profile — and reports the maximum absolute deviation after peak-normalisation as an honest measure of how far the smooth-step approximation deviates from the exact B-spline.

In [ ]:
from shape_blend_splines.basis import bspline_basis, sbs_basis, smooth_step_at

t_th = np.linspace(0.0, 1.0, 500)

# ── Part 1: exact identity for sbs_basis ────────────────────────────────
a, b = 0.2, 0.8
sigma = (b - a) / 2.0
S_a = smooth_step_at(t_th, centre=a, half_width=sigma, rising=False)
S_b = smooth_step_at(t_th, centre=b, half_width=sigma, rising=False)
B_manual = np.clip(S_b - S_a, 0.0, None)
B_sbs    = sbs_basis(t_th, a, b)
err1 = np.max(np.abs(B_sbs - B_manual))
print(f"Part 1 — sbs_basis vs manual step-difference (a={a}, b={b}):")
print(f"  max |sbs_basis - (S_b - S_a)| = {err1:.2e}  (should be < 1e-12)\n")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.0))

ax = axes[0]
ax.plot(t_th, S_a, "--", color="tomato",    lw=1.8, label="$S_a(t)$ (falling step at a=0.2)")
ax.plot(t_th, S_b, "--", color="seagreen",  lw=1.8, label="$S_b(t)$ (falling step at b=0.8)")
ax.plot(t_th, B_sbs, color="steelblue",     lw=2.5, label="$B_{a,b}(t) = S_b - S_a$")
ax.axvline(a, color="tomato",   lw=0.8, ls=":", alpha=0.7)
ax.axvline(b, color="seagreen", lw=0.8, ls=":", alpha=0.7)
ax.set_title("SBS step-difference: $B_{a,b}(t) = S_b(t) - S_a(t)$")
ax.set_xlabel("t"); ax.set_ylabel("value")
ax.legend(fontsize=9); ax.grid(alpha=0.2)

# ── Part 2: cubic B-spline vs sbs_basis structural parallel ─────────────
p, n_ctrl = 3, 6
inner_knots = np.linspace(0.0, 1.0, n_ctrl - p + 1)
knots = np.concatenate([np.zeros(p), inner_knots, np.ones(p)])

ax = axes[1]
tab_c = plt.cm.tab10.colors
max_errs = []
for i in range(n_ctrl):
    a_i = float(knots[i])
    b_i = float(knots[i + p + 1])
    N = bspline_basis(i, p, t_th, knots)
    peak_N = N.max()
    if peak_N < 1e-10 or np.isclose(a_i, b_i):
        continue
    N_norm = N / peak_N

    B = sbs_basis(t_th, a_i, b_i)
    peak_B = B.max()
    if peak_B > 1e-10:
        B_norm = B / peak_B
    else:
        B_norm = B

    err_i = np.max(np.abs(N_norm - B_norm))
    max_errs.append(err_i)

    c = tab_c[i % len(tab_c)]
    ax.plot(t_th, N_norm, "-",  color=c, lw=2.0, alpha=0.85,
            label=f"$N_{{{i},{p}}}$ (Cox–de Boor)")
    ax.plot(t_th, B_norm, ":", color=c, lw=1.5, alpha=0.85,
            label=f"sbs basis $B_{{{a_i:.2f},{b_i:.2f}}}$")

ax.set_title(f"Peak-normalised: cubic B-spline vs SBS step-diff\n"
             f"Mean max-err = {np.mean(max_errs):.3f}  (shape differs; "
             f"B-spline uses exact divided-diff, SBS uses smooth $T_n$)")
ax.set_xlabel("t"); ax.set_ylabel("normalised value")
ax.grid(alpha=0.2)

print("Part 2 — peak-normalised comparison, cubic B-spline vs sbs_basis:")
for i_idx, err in enumerate(max_errs):
    print(f"  basis {i_idx}: max err = {err:.4f}")
print(f"  Mean max deviation = {np.mean(max_errs):.4f}")
print("\nNote: SBS uses a smooth-step approximation, not the exact divided-diff cumulative.")
print("The structural parallel (non-negativity, locality, bell shape) is the key connection.")

plt.tight_layout()
plt.show()

## 5) Control-point workflow

The package also includes a familiar control-point interface for open and closed spline-style design tasks.  The `ControlPointSpline` uses cubic Hermite (Catmull–Rom) interpolation internally.

In [ ]:
control_pts = np.array([
    [0.0, 0.0],
    [1.0, 1.8],
    [3.0, 0.8],
    [4.0, 2.4],
    [5.5, 0.2],
])

open_cp   = ControlPointSpline(control_pts, locality=2.0)
closed_cp = ControlPointSpline(control_pts, locality=2.0, closed=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].plot(*open_cp.evaluate(np.linspace(0.0, 1.0, 700)).T,
             color="steelblue", lw=2.5)
axes[0].plot(control_pts[:, 0], control_pts[:, 1], "o--", color="tomato", lw=1.2)
axes[0].set_title("Open control-point spline")
axes[0].set_aspect("equal")

closed_poly = np.vstack([control_pts, control_pts[:1]])
axes[1].plot(*closed_cp.evaluate(np.linspace(0.0, 1.0, 700, endpoint=False)).T,
             color="seagreen", lw=2.5)
axes[1].plot(closed_poly[:, 0], closed_poly[:, 1], "o--", color="tomato", lw=1.2)
axes[1].set_title("Closed control-point spline")
axes[1].set_aspect("equal")

for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("y")
plt.tight_layout()
plt.show()